# Notebook 06: Residual Characterization & Diagnostic Analysis

## 1. Objectives & Methodological Framing
This notebook provides a rigorous post-hoc characterization of estimation residuals from the frozen **Group C Initial-SoC estimator** (12 summary features, seed 42).

### Primary Scientific Objective
> *"Is the performance degradation from approximately 4.30% validation RMSE (-10°C) to approximately 10.15% cold/aged test RMSE (-20°C) associated with a systematic residual structure that can be linked to observable operating variables?"*

### Strict Methodological Guardrails
1. **Completely Frozen Model & Scaler:** Evaluates `results/models/ablation_group_c_seed_42.pt` with `data/ablation_scaler_group_c.pkl`. No retraining, fine-tuning, or parameter modification is performed.
2. **Diagnostic Only — No Correction Applied:** Any regression lines or trends fit to residuals are strictly **descriptive visual diagnostics**. No correction model, linear adjustment, or residual compensation is applied to modify predictions in this notebook.
3. **Test-Set Discipline:** Validation (-10°C) is the sole split used for driving insights or formulating future methodological hypotheses. The held-out Test set (-20°C) is evaluated strictly post-hoc to characterize its empirical error distribution. Zero test patterns are used to tune or select parameters.
4. **No Causal Overclaiming:** Distinguish observed empirical patterns (e.g., mean residual shift, boundary error spikes) from unproven physical hypotheses (polarization, phase-change kinetics, capacity fade).

In [1]:
import os
import sys
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Selected compute device: {device}")
print(f"PyTorch Version:         {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Runtime Version:    {torch.version.cuda}")
    print(f"GPU Hardware:            {torch.cuda.get_device_name(0)}")

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Selected compute device: cuda
PyTorch Version:         2.6.0+cu124
CUDA Runtime Version:    12.4
GPU Hardware:            NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Load Frozen Model, Scalers, and 50s Datasets
We load the designated frozen Group C model (`ablation_group_c_seed_42.pt`), its training-only feature scaler (`ablation_scaler_group_c.pkl`), the channel normalization parameters (`initial_soc_scaler.npz`), and the 50s nominal dataset (`initial_soc_datasets.pkl`).

In [2]:
PROJECT_ROOT = Path(r"c:\Users\sheok\OneDrive\Desktop\New folder (2)\Major Project")
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = PROJECT_ROOT / "results"
MODELS_DIR   = RESULTS_DIR / "models"
PLOTS_DIR    = RESULTS_DIR / "plots"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Model Architecture (Group C: 12 Inputs)
class SummaryMLP(nn.Module):
    def __init__(self, in_dim=12):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

# Load Frozen Model
MODEL_PATH = MODELS_DIR / "ablation_group_c_seed_42.pt"
model = SummaryMLP(in_dim=12).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f"Loaded frozen Group C MLP checkpoint from: {MODEL_PATH.name}")

# Load Frozen Group C Scaler (Fitted on training data only)
SCALER_PATH = DATA_DIR / "ablation_scaler_group_c.pkl"
with open(SCALER_PATH, "rb") as f:
    group_c_scaler = pickle.load(f)
print(f"Loaded frozen Group C scaler from: {SCALER_PATH.name}")

# Load Raw Sequence Normalization Scaler
RAW_SCALER_PATH = DATA_DIR / "initial_soc_scaler.npz"
raw_scaler_data = np.load(RAW_SCALER_PATH)
feat_min   = raw_scaler_data["feat_min"]
feat_range = raw_scaler_data["feat_range"]

# Load 50s Nominal Datasets
DATASET_PATH = DATA_DIR / "initial_soc_datasets.pkl"
with open(DATASET_PATH, "rb") as f:
    datasets = pickle.load(f)

dur = 50
X_val_norm  = datasets[dur]["X_val"]   # (1045, 500, 3) normalized [0, 1]
y_val       = datasets[dur]["y_val"]   # (1045,) normalized [0, 1]
X_test_norm = datasets[dur]["X_test"]  # (871, 500, 3)
y_test      = datasets[dur]["y_test"]  # (871,)

print(f"Loaded 50s Nominal Sequences: Validation N={len(X_val_norm)}, Test N={len(X_test_norm)}")


Loaded frozen Group C MLP checkpoint from: ablation_group_c_seed_42.pt
Loaded frozen Group C scaler from: ablation_scaler_group_c.pkl
Loaded 50s Nominal Sequences: Validation N=1045, Test N=871


## 3. Feature Extraction (Normalized & Physical Units) & Model Inference

We extract the 12 Group C summary features:
1. **Normalized Features:** Formatted as input to the frozen scaler and PyTorch model.
2. **Physical Features (Engineering Units):** Expressed in Volts (V), Amperes (A), and Degrees Celsius (°C) for physical correlation and quantile analysis.

**Feature List (Group C, 12 Features):**
- Initial operating point: `V0` (V), `I0` (A), `T0` (°C)
- Window endpoints: `Vend` (V), `Iend` (A), `Tend` (°C)
- Trajectory deltas: `dV` (V), `dI` (A), `dT` (°C)
- Aggregate trajectory statistics: `meanV` (V), `meanI` (A), `stdV` (V)

In [3]:
def extract_group_c_norm(X_seq_norm):
    """Extracts 12 Group C summary features in normalized channel space."""
    V0 = X_seq_norm[:, 0, 0]
    I0 = X_seq_norm[:, 0, 1]
    T0 = X_seq_norm[:, 0, 2]
    Vend = X_seq_norm[:, -1, 0]
    Iend = X_seq_norm[:, -1, 1]
    Tend = X_seq_norm[:, -1, 2]
    dV = Vend - V0
    dI = Iend - I0
    dT = Tend - T0
    meanV = np.mean(X_seq_norm[:, :, 0], axis=1)
    meanI = np.mean(X_seq_norm[:, :, 1], axis=1)
    stdV  = np.std(X_seq_norm[:, :, 0], axis=1)
    return np.column_stack([V0, I0, T0, Vend, Iend, Tend, dV, dI, dT, meanV, meanI, stdV])

def extract_group_c_phys(X_seq_norm):
    """Extracts 12 Group C summary features converted to physical units (V, A, °C)."""
    V_phys = X_seq_norm[:, :, 0] * feat_range[0] + feat_min[0]
    I_phys = X_seq_norm[:, :, 1] * feat_range[1] + feat_min[1]
    T_phys = X_seq_norm[:, :, 2] * feat_range[2] + feat_min[2]
    
    return pd.DataFrame({
        'V0': V_phys[:, 0],
        'I0': I_phys[:, 0],
        'T0': T_phys[:, 0],
        'Vend': V_phys[:, -1],
        'Iend': I_phys[:, -1],
        'Tend': T_phys[:, -1],
        'dV': V_phys[:, -1] - V_phys[:, 0],
        'dI': I_phys[:, -1] - I_phys[:, 0],
        'dT': T_phys[:, -1] - T_phys[:, 0],
        'meanV': np.mean(V_phys, axis=1),
        'meanI': np.mean(I_phys, axis=1),
        'stdV': np.std(V_phys, axis=1)
    })

def predict_initial_soc(feats_norm, scaler, model, batch_size=64):
    """Runs forward pass through frozen scaler and model, returning physical % SoC."""
    feats_scaled = scaler.transform(feats_norm)
    X_tensor = torch.tensor(feats_scaled, dtype=torch.float32)
    loader = torch.utils.data.DataLoader(X_tensor, batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)
            p = model(xb).squeeze().cpu().numpy()
            preds.append(p)
    return np.concatenate(preds) * 100.0

# Extract features
feats_val_norm  = extract_group_c_norm(X_val_norm)
feats_test_norm = extract_group_c_norm(X_test_norm)

df_phys_val  = extract_group_c_phys(X_val_norm)
df_phys_test = extract_group_c_phys(X_test_norm)

# Reference targets in percentage SoC
y_val_phys  = y_val * 100.0
y_test_phys = y_test * 100.0

df_phys_val['ref_SoC']  = y_val_phys
df_phys_test['ref_SoC'] = y_test_phys

# Run model predictions
pred_val  = predict_initial_soc(feats_val_norm, group_c_scaler, model)
pred_test = predict_initial_soc(feats_test_norm, group_c_scaler, model)

# Residual definition: e = SoC_pred - SoC_ref
res_val  = pred_val - y_val_phys
res_test = pred_test - y_test_phys

df_phys_val['residual']  = res_val
df_phys_test['residual'] = res_test

print(f"Computed predictions and residuals for N={len(res_val)} Validation and N={len(res_test)} Test samples.")


Computed predictions and residuals for N=1045 Validation and N=871 Test samples.


## 4. Residual Definition, Summary Statistics & Exact MSE Decomposition

### Mathematical Decomposition
We define the estimation residual as:
$$e = \text{SoC}_{\text{pred}} - \text{SoC}_{\text{ref}} \quad (\% \text{ SoC})$$

The Mean Squared Error (MSE) decomposes exactly into squared mean signed error (systematic offset) and residual variance (unexplained dispersion):
$$\text{MSE} = \frac{1}{N}\sum_{i=1}^N e_i^2 = \bar{e}^2 + \sigma_e^2$$
where:
- $\bar{e} = \frac{1}{N}\sum_{i=1}^N e_i$ is the **Mean Signed Error (Bias)**
- $\sigma_e^2 = \frac{1}{N}\sum_{i=1}^N (e_i - \bar{e})^2$ is the **Residual Variance**

> **Methodological Interpretation:** The term $\bar{e}^2 / \text{MSE}$ measures the mathematical contribution of the systematic mean residual component to total MSE. It must **not** be causally attributed to capacity fade, but simply describes the fraction of quadratic error attributable to a fixed DC offset.

In [4]:
def compute_residual_metrics(residuals, split_name):
    N = len(residuals)
    mean_e = np.mean(residuals)
    mae = np.mean(np.abs(residuals))
    mse = np.mean(residuals**2)
    rmse = np.sqrt(mse)
    std_e = np.std(residuals)
    
    bias_sq = mean_e**2
    variance = std_e**2
    bias_sq_pct = (bias_sq / mse) * 100.0
    var_pct = (variance / mse) * 100.0
    
    q25 = np.percentile(residuals, 25)
    median = np.median(residuals)
    q75 = np.percentile(residuals, 75)
    min_val = np.min(residuals)
    max_val = np.max(residuals)
    
    return {
        'Split': split_name,
        'N': N,
        'Mean_Signed_Error': mean_e,
        'MAE': mae,
        'RMSE': rmse,
        'Std_Residual': std_e,
        'MSE': mse,
        'Bias_Sq': bias_sq,
        'Variance': variance,
        'Bias_Sq_Share_Pct': bias_sq_pct,
        'Variance_Share_Pct': var_pct,
        'Min': min_val,
        'Q25': q25,
        'Median': median,
        'Q75': q75,
        'Max': max_val
    }

val_stats  = compute_residual_metrics(res_val, "Validation (-10°C)")
test_stats = compute_residual_metrics(res_test, "Test (-20°C Post-Hoc)")

df_res_summary = pd.DataFrame([val_stats, test_stats])
df_res_summary.to_csv(RESULTS_DIR / "residual_summary.csv", index=False)

print("=== RESIDUAL SUMMARY STATISTICS & MSE DECOMPOSITION ===")
cols_display = ['Split', 'N', 'Mean_Signed_Error', 'MAE', 'RMSE', 'Std_Residual', 'Bias_Sq_Share_Pct', 'Variance_Share_Pct']
print(df_res_summary[cols_display].to_string(index=False))

print("\nDetailed Quartiles:")
print(df_res_summary[['Split', 'Min', 'Q25', 'Median', 'Q75', 'Max']].to_string(index=False))


=== RESIDUAL SUMMARY STATISTICS & MSE DECOMPOSITION ===
                Split    N  Mean_Signed_Error      MAE      RMSE  Std_Residual  Bias_Sq_Share_Pct  Variance_Share_Pct
   Validation (-10°C) 1045           -2.14656 3.292793  4.300521      3.726494          24.914022           75.085983
Test (-20°C Post-Hoc)  871           -6.64630 7.880585 10.154443      7.677199          42.839832           57.160164

Detailed Quartiles:
                Split        Min        Q25    Median       Q75       Max
   Validation (-10°C) -17.618286  -4.530338 -1.883011  0.213654 13.570427
Test (-20°C Post-Hoc) -32.845520 -11.660955 -5.734024 -1.318077 14.080460


## 5. Residual Distribution Plots (Histograms & KDEs)
We visualize the empirical error distributions for Validation (-10°C) and Test (-20°C Post-Hoc) to assess skewness, modality, and mean bias.

In [5]:
# Plot 1: Validation Residual Histogram
plt.figure(figsize=(7, 5))
counts, bins, _ = plt.hist(res_val, bins=40, density=True, alpha=0.6, color='steelblue', edgecolor='black', label='Residual Empirical Dist')
kde_x = np.linspace(res_val.min() - 2, res_val.max() + 2, 300)
kde = stats.gaussian_kde(res_val)
plt.plot(kde_x, kde(kde_x), 'navy', lw=2, label='Kernel Density Estimate (KDE)')

plt.axvline(0, color='black', linestyle=':', lw=1.5, label='Zero Error ($e=0$)')
plt.axvline(val_stats['Mean_Signed_Error'], color='crimson', linestyle='--', lw=2, 
            label=f"Mean Signed Error ({val_stats['Mean_Signed_Error']:.2f}%)")
plt.axvline(val_stats['Mean_Signed_Error'] - val_stats['Std_Residual'], color='gray', linestyle='-.', lw=1, label='$\pm 1\sigma$ Interval')
plt.axvline(val_stats['Mean_Signed_Error'] + val_stats['Std_Residual'], color='gray', linestyle='-.', lw=1)

plt.title('Initial-SoC Residual Distribution: Validation (-10°C)\n(Group C 12-Feature MLP, Seed 42)', fontsize=12)
plt.xlabel('Residual $e = \text{SoC}_{\text{pred}} - \text{SoC}_{\text{ref}}$ (% SoC)', fontsize=11)
plt.ylabel('Probability Density', fontsize=11)
plt.legend(loc='upper right', fontsize=9)
plt.grid(True, alpha=0.3)

# Text box with summary
text_str = f"$N$ = {val_stats['N']}\nMean = {val_stats['Mean_Signed_Error']:.2f}%\nStd = {val_stats['Std_Residual']:.2f}%\nMAE = {val_stats['MAE']:.2f}%\nRMSE = {val_stats['RMSE']:.2f}%\nBias² Share = {val_stats['Bias_Sq_Share_Pct']:.1f}%"
plt.text(0.03, 0.95, text_str, transform=plt.gca().transAxes, fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_residual_hist_validation.png", dpi=300)
plt.close()
print(f"Saved Validation Histogram to: {PLOTS_DIR / '06_residual_hist_validation.png'}")

# Plot 2: Test Residual Histogram (Post-Hoc)
plt.figure(figsize=(7, 5))
plt.hist(res_test, bins=40, density=True, alpha=0.6, color='indianred', edgecolor='black', label='Residual Empirical Dist')
kde_x_test = np.linspace(res_test.min() - 2, res_test.max() + 2, 300)
kde_test = stats.gaussian_kde(res_test)
plt.plot(kde_x_test, kde_test(kde_x_test), 'darkred', lw=2, label='Kernel Density Estimate (KDE)')

plt.axvline(0, color='black', linestyle=':', lw=1.5, label='Zero Error ($e=0$)')
plt.axvline(test_stats['Mean_Signed_Error'], color='blue', linestyle='--', lw=2, 
            label=f"Mean Signed Error ({test_stats['Mean_Signed_Error']:.2f}%)")
plt.axvline(test_stats['Mean_Signed_Error'] - test_stats['Std_Residual'], color='gray', linestyle='-.', lw=1, label='$\pm 1\sigma$ Interval')
plt.axvline(test_stats['Mean_Signed_Error'] + test_stats['Std_Residual'], color='gray', linestyle='-.', lw=1)

plt.title('Initial-SoC Residual Distribution: Test (-20°C Post-Hoc Diagnostic)\n(Group C 12-Feature MLP, Seed 42)', fontsize=12)
plt.xlabel('Residual $e = \text{SoC}_{\text{pred}} - \text{SoC}_{\text{ref}}$ (% SoC)', fontsize=11)
plt.ylabel('Probability Density', fontsize=11)
plt.legend(loc='upper right', fontsize=9)
plt.grid(True, alpha=0.3)

text_str_test = f"$N$ = {test_stats['N']} (Post-Hoc)\nMean = {test_stats['Mean_Signed_Error']:.2f}%\nStd = {test_stats['Std_Residual']:.2f}%\nMAE = {test_stats['MAE']:.2f}%\nRMSE = {test_stats['RMSE']:.2f}%\nBias² Share = {test_stats['Bias_Sq_Share_Pct']:.1f}%"
plt.text(0.03, 0.95, text_str_test, transform=plt.gca().transAxes, fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_residual_hist_test.png", dpi=300)
plt.close()
print(f"Saved Test Histogram to: {PLOTS_DIR / '06_residual_hist_test.png'}")


Saved Validation Histogram to: c:\Users\sheok\OneDrive\Desktop\New folder (2)\Major Project\results\plots\06_residual_hist_validation.png
Saved Test Histogram to: c:\Users\sheok\OneDrive\Desktop\New folder (2)\Major Project\results\plots\06_residual_hist_test.png


<cell_10>:11: SyntaxWarning: invalid escape sequence '\p'
<cell_10>:39: SyntaxWarning: invalid escape sequence '\p'


## 6. Residuals vs. Observable Features & Bivariate Correlations

We investigate whether residual errors correlate with observable operating variables:
- Pearson correlation $r$ measures linear association.
- Spearman rank correlation $\rho$ measures monotonic association.
- Scatter plots with simple ordinary least squares (OLS) lines provide descriptive visualization of trends across Validation and Test.

In [6]:
feature_cols = ['V0', 'I0', 'T0', 'Vend', 'Iend', 'Tend', 'dV', 'dI', 'dT', 'meanV', 'meanI', 'stdV', 'ref_SoC']

corrs = []
for col in feature_cols:
    r_val, p_val = stats.pearsonr(df_phys_val[col], res_val)
    rho_val, p_rho_val = stats.spearmanr(df_phys_val[col], res_val)
    
    r_test, p_test = stats.pearsonr(df_phys_test[col], res_test)
    rho_test, p_rho_test = stats.spearmanr(df_phys_test[col], res_test)
    
    corrs.append({
        'Observable': col,
        'Val_Pearson_r': r_val,
        'Val_Spearman_rho': rho_val,
        'Test_Pearson_r': r_test,
        'Test_Spearman_rho': rho_test
    })

df_corrs = pd.DataFrame(corrs)
df_corrs.to_csv(RESULTS_DIR / "residual_feature_correlations.csv", index=False)
print("=== BIVARIATE CORRELATIONS: RESIDUAL VS OBSERVABLE FEATURES ===")
print(df_corrs.to_string(index=False))

# Multi-Panel Scatter Grid for Primary Observables
plot_features = [
    ('V0', 'Initial Voltage $V_0$ (V)'),
    ('meanV', 'Mean Voltage $\\mathrm{mean}(V)$ (V)'),
    ('dV', 'Voltage Delta $\\Delta V$ (V)'),
    ('I0', 'Initial Current $I_0$ (A)'),
    ('meanI', 'Mean Current $\\mathrm{mean}(I)$ (A)'),
    ('stdV', 'Voltage Volatility $\\mathrm{std}(V)$ (V)')
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey=True)
axes = axes.flatten()

for i, (col, label) in enumerate(plot_features):
    ax = axes[i]
    
    # Validation scatter & linear descriptive fit
    ax.scatter(df_phys_val[col], res_val, alpha=0.3, s=12, c='steelblue', label='Val (-10°C)')
    slope_v, intercept_v, r_v, _, _ = stats.linregress(df_phys_val[col], res_val)
    x_v = np.linspace(df_phys_val[col].min(), df_phys_val[col].max(), 100)
    ax.plot(x_v, slope_v * x_v + intercept_v, color='navy', lw=1.8, linestyle='-', label=f'Val Fit ($r={r_v:.2f}$)')
    
    # Test scatter & linear descriptive fit (Post-Hoc)
    ax.scatter(df_phys_test[col], res_test, alpha=0.3, s=12, c='indianred', label='Test (-20°C Post-Hoc)')
    slope_t, intercept_t, r_t, _, _ = stats.linregress(df_phys_test[col], res_test)
    x_t = np.linspace(df_phys_test[col].min(), df_phys_test[col].max(), 100)
    ax.plot(x_t, slope_t * x_t + intercept_t, color='darkred', lw=1.8, linestyle='--', label=f'Test Fit ($r={r_t:.2f}$)')
    
    ax.axhline(0, color='black', linestyle=':', lw=1)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel(label, fontsize=10)
    if i % 3 == 0:
        ax.set_ylabel('Residual (% SoC)', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='best')

plt.suptitle('Initial-SoC Residuals vs. Physical Observables\n(Trend lines are descriptive visual diagnostics only; no corrections applied)', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_residual_feature_relationships.png", dpi=300)
plt.close()
print(f"Saved Multi-Panel Feature Scatter Grid to: {PLOTS_DIR / '06_residual_feature_relationships.png'}")


=== BIVARIATE CORRELATIONS: RESIDUAL VS OBSERVABLE FEATURES ===
Observable  Val_Pearson_r  Val_Spearman_rho  Test_Pearson_r  Test_Spearman_rho
        V0       0.103737          0.073524        0.162066           0.164663
        I0       0.196030          0.184465        0.252920           0.293669
        T0      -0.269129         -0.284686       -0.052506          -0.081692
      Vend       0.328171          0.313750        0.496924           0.549669
      Iend       0.514042          0.570494        0.557736           0.701931
      Tend      -0.352398         -0.362231       -0.178533          -0.207749
        dV       0.277646          0.273469        0.333954           0.317725
        dI       0.245483          0.255049        0.244505           0.260525
        dT      -0.418998         -0.410188       -0.559616          -0.590289
     meanV       0.231696          0.195622        0.387286           0.381182
     meanI       0.589210          0.575443        0.661135        

## 7. Temperature / SoC Interaction & Boundary Analysis

We examine the empirical structure of residuals across the state-of-charge spectrum:
1. **Non-Parametric Trend with SoC:** Testing whether residuals vary non-linearly with SoC.
2. **10% Interval SoC Bin Breakdown:** Quantifying count, mean signed error, MAE, and RMSE.
3. **Extreme Cold (-20°C) Boundary Behavior:** Evaluating the previously noted elevation in the 90–100% bin and verifying the absence of test samples below 30% SoC.

In [7]:
# Dual-Panel Residual vs SoC with Non-Parametric Trend
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Left: Validation (-10°C)
ax1.scatter(y_val_phys, res_val, alpha=0.35, s=15, c='steelblue', label='Validation Samples (-10°C)')
ax1.axhline(0, color='black', linestyle=':', lw=1.5, label='Zero Error')
ax1.axhline(val_stats['Mean_Signed_Error'], color='navy', linestyle='--', lw=1.5, 
            label=f"Mean Signed Error ({val_stats['Mean_Signed_Error']:.2f}%)")

# Binned non-parametric moving average trend
sort_idx_v = np.argsort(y_val_phys)
soc_sorted_v = y_val_phys[sort_idx_v]
res_sorted_v = res_val[sort_idx_v]
trend_v = pd.Series(res_sorted_v).rolling(window=75, min_periods=25, center=True).mean()
ax1.plot(soc_sorted_v, trend_v, color='darkorange', lw=2.5, label='Non-Parametric Trend (Rolling Mean)')

ax1.set_title('Residual vs. Reference SoC: Validation (-10°C)', fontsize=11)
ax1.set_xlabel('Reference SoC (%)', fontsize=10)
ax1.set_ylabel('Residual $e = \text{SoC}_{\text{pred}} - \text{SoC}_{\text{ref}}$ (% SoC)', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower left', fontsize=9)

# Right: Test (-20°C Post-Hoc)
ax2.scatter(y_test_phys, res_test, alpha=0.35, s=15, c='indianred', label='Test Samples (-20°C Post-Hoc)')
ax2.axhline(0, color='black', linestyle=':', lw=1.5, label='Zero Error')
ax2.axhline(test_stats['Mean_Signed_Error'], color='darkred', linestyle='--', lw=1.5, 
            label=f"Mean Signed Error ({test_stats['Mean_Signed_Error']:.2f}%)")

sort_idx_t = np.argsort(y_test_phys)
soc_sorted_t = y_test_phys[sort_idx_t]
res_sorted_t = res_test[sort_idx_t]
trend_t = pd.Series(res_sorted_t).rolling(window=75, min_periods=25, center=True).mean()
ax2.plot(soc_sorted_t, trend_t, color='blue', lw=2.5, label='Non-Parametric Trend (Rolling Mean)')

ax2.set_title('Residual vs. Reference SoC: Test (-20°C Post-Hoc Diagnostic)', fontsize=11)
ax2.set_xlabel('Reference SoC (%)', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='lower left', fontsize=9)

# Highlight missing test range below 30%
ax2.axvspan(0, 30, color='lightgray', alpha=0.3, label='No Test Data (<30% SoC)')

plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_residual_vs_soc.png", dpi=300)
plt.close()
print(f"Saved Residual vs. SoC Plot to: {PLOTS_DIR / '06_residual_vs_soc.png'}")

# 10% SoC Interval Breakdown Table
bins = np.arange(0, 110, 10)
bin_labels = [f'{bins[i]}-{bins[i+1]}%' for i in range(len(bins)-1)]

records = []
for i in range(len(bins)-1):
    low, high = bins[i], bins[i+1]
    mv = (y_val_phys >= low) & (y_val_phys < high)
    mt = (y_test_phys >= low) & (y_test_phys < high)
    
    nv = int(np.sum(mv))
    nt = int(np.sum(mt))
    
    records.append({
        'SoC_Bin': bin_labels[i],
        'Val_Count': nv,
        'Val_Mean_Residual': np.mean(res_val[mv]) if nv > 0 else np.nan,
        'Val_MAE': mean_absolute_error(y_val_phys[mv], pred_val[mv]) if nv > 0 else np.nan,
        'Val_RMSE': np.sqrt(mean_squared_error(y_val_phys[mv], pred_val[mv])) if nv > 0 else np.nan,
        'Test_Count': nt,
        'Test_Mean_Residual': np.mean(res_test[mt]) if nt > 0 else np.nan,
        'Test_MAE': mean_absolute_error(y_test_phys[mt], pred_test[mt]) if nt > 0 else np.nan,
        'Test_RMSE': np.sqrt(mean_squared_error(y_test_phys[mt], pred_test[mt])) if nt > 0 else np.nan,
    })

df_bins = pd.DataFrame(records)
df_bins.to_csv(RESULTS_DIR / "residual_soc_bins.csv", index=False)
print("=== RESIDUAL ERROR BREAKDOWN BY 10% SOC INTERVALS ===")
print(df_bins.to_string(index=False))

# Plot 5: Grouped Bar Chart of SoC Bins
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
x_pos = np.arange(len(df_bins))
bar_w = 0.35

# Top: Mean Signed Error (Directional Bias)
ax_top.bar(x_pos - bar_w/2, df_bins['Val_Mean_Residual'].fillna(0), width=bar_w, label='Val Mean Residual (-10°C)', color='royalblue')
ax_top.bar(x_pos + bar_w/2, df_bins['Test_Mean_Residual'].fillna(0), width=bar_w, label='Test Mean Residual (-20°C Post-Hoc)', color='indianred')
ax_top.axhline(0, color='black', linestyle=':', lw=1.2)
ax_top.set_ylabel('Mean Residual (% SoC)', fontsize=10)
ax_top.set_title('Directional Mean Residual Bias across 10% SoC Intervals', fontsize=11)
ax_top.legend(loc='best', fontsize=9)
ax_top.grid(True, alpha=0.3, axis='y')

# Bottom: RMSE Comparison
ax_bot.bar(x_pos - bar_w/2, df_bins['Val_RMSE'].fillna(0), width=bar_w, label='Val RMSE (-10°C)', color='royalblue')
ax_bot.bar(x_pos + bar_w/2, df_bins['Test_RMSE'].fillna(0), width=bar_w, label='Test RMSE (-20°C Post-Hoc)', color='indianred')
ax_bot.set_ylabel('RMSE (% SoC)', fontsize=10)
ax_bot.set_title('Root Mean Squared Error (RMSE) across 10% SoC Intervals', fontsize=11)
ax_bot.set_xticks(x_pos)
ax_bot.set_xticklabels(df_bins['SoC_Bin'])
ax_bot.set_xlabel('State-of-Charge Interval', fontsize=10)
ax_bot.legend(loc='upper left', fontsize=9)
ax_bot.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_residual_soc_bins.png", dpi=300)
plt.close()
print(f"Saved SoC Bins Bar Chart to: {PLOTS_DIR / '06_residual_soc_bins.png'}")


Saved Residual vs. SoC Plot to: c:\Users\sheok\OneDrive\Desktop\New folder (2)\Major Project\results\plots\06_residual_vs_soc.png
=== RESIDUAL ERROR BREAKDOWN BY 10% SOC INTERVALS ===
SoC_Bin  Val_Count  Val_Mean_Residual  Val_MAE  Val_RMSE  Test_Count  Test_Mean_Residual  Test_MAE  Test_RMSE
  0-10%          0                NaN      NaN       NaN           0                 NaN       NaN        NaN
 10-20%          0                NaN      NaN       NaN           0                 NaN       NaN        NaN
 20-30%         45           3.387516 4.805447  5.634505           0                 NaN       NaN        NaN
 30-40%        123          -1.883214 3.478520  4.157346          40            6.967430  7.131834   7.940179
 40-50%        134          -2.387130 3.276615  4.191724         137           -5.692956  6.706629   8.347262
 50-60%        143          -2.399926 2.920818  4.000495         135           -6.849391  7.325796   9.244886
 60-70%        154          -2.288478 2.875254

## 8. Feature Quantile Diagnostics

To determine whether error patterns vary monotonically or non-linearly across physical features, we divide samples into quartiles ($Q_1$ to $Q_4$) for top candidate observables:
- Initial Voltage ($V_0$)
- Window Mean Voltage ($\text{mean}(V)$)
- Window Voltage Delta ($\Delta V$)
- Initial Current ($I_0$)
- Window Mean Current ($\text{mean}(I)$)
- Voltage Volatility ($\text{std}(V)$)
- Reference SoC ($\text{ref\_SoC}$)

In [8]:
def compute_quantile_diagnostics(df_features, feature_name, q=4):
    """Divides an observable into quantiles and computes error statistics per bin."""
    try:
        bins, bin_edges = pd.qcut(df_features[feature_name], q=q, retbins=True, duplicates='drop')
    except Exception:
        return pd.DataFrame()
    
    df_temp = df_features[[feature_name, 'residual', 'ref_SoC']].copy()
    df_temp['bin'] = bins
    
    records = []
    for b, grp in df_temp.groupby('bin', observed=True):
        cnt = len(grp)
        res = grp['residual'].values
        records.append({
            'Observable': feature_name,
            'Quantile_Range': str(b),
            'Count': cnt,
            'Median_Value': grp[feature_name].median(),
            'Mean_Residual': np.mean(res),
            'MAE': np.mean(np.abs(res)),
            'RMSE': np.sqrt(np.mean(res**2))
        })
    return pd.DataFrame(records)

diag_features = ['V0', 'meanV', 'dV', 'I0', 'meanI', 'stdV', 'ref_SoC']

print("=== QUANTILE DIAGNOSTICS: VALIDATION (-10°C) ===")
for feat in diag_features:
    q_df = compute_quantile_diagnostics(df_phys_val, feat, q=4)
    print(f"\n--- {feat} (Validation) ---")
    print(q_df[['Quantile_Range', 'Count', 'Median_Value', 'Mean_Residual', 'MAE', 'RMSE']].to_string(index=False))

print("\n=== QUANTILE DIAGNOSTICS: TEST (-20°C POST-HOC) ===")
for feat in diag_features:
    q_df = compute_quantile_diagnostics(df_phys_test, feat, q=4)
    print(f"\n--- {feat} (Test Post-Hoc) ---")
    print(q_df[['Quantile_Range', 'Count', 'Median_Value', 'Mean_Residual', 'MAE', 'RMSE']].to_string(index=False))


=== QUANTILE DIAGNOSTICS: VALIDATION (-10°C) ===

--- V0 (Validation) ---
Quantile_Range  Count  Median_Value  Mean_Residual      MAE     RMSE
(2.498, 3.383]    262      3.257035      -3.252278 4.011298 4.900164
(3.383, 3.547]    261      3.465310      -0.940898 3.364373 4.294627
 (3.547, 3.72]    261      3.634370      -2.112374 2.718552 3.641590
 (3.72, 4.173]    261      3.832050      -2.276453 3.074196 4.270851

--- meanV (Validation) ---
             Quantile_Range  Count  Median_Value  Mean_Residual      MAE     RMSE
(2.7350000000000003, 3.381]    262      3.263091      -3.687978 4.393450 5.282004
             (3.381, 3.536]    261      3.451612      -1.003684 3.139701 4.072924
             (3.536, 3.701]    261      3.628921      -2.400845 2.996583 4.038078
             (3.701, 4.074]    261      3.790058      -1.487826 2.637221 3.625952

--- dV (Validation) ---
    Quantile_Range  Count  Median_Value  Mean_Residual      MAE     RMSE
 (-0.966, -0.0849]    263      -0.20266      

## 9. Diagnostic Findings, Calibration Structure & Scientific Takeaways

### 1. Tripartite Distinction: Observation vs. Interpretation vs. Causation

- **Observed Statistical Associations:**
  1. **Mean Residual Offset:** On Validation (-10°C), Mean Signed Error is **-2.15% SoC** (accounting for 24.9% of total MSE; Variance = 75.1%). On Test (-20°C Post-Hoc), Mean Signed Error drops to **-6.65% SoC**, accounting for **42.8% of total MSE** (Variance = 57.2%).
  2. **Correlation with Discharge Current:** Residuals exhibit a strong positive correlation with window current observables (`meanI`: $r = +0.59$ Val, $+0.66$ Test; `Iend`: $r = +0.51$ Val, $+0.56$ Test).
  3. **Error Stratification across Current Quartiles:** Under near-rest conditions ($I \approx 0$), Test RMSE drops to **5.43%** (bias **-0.70%**). Under heavy discharge ($Q_1$, median $-2.48\text{ A}$), Test RMSE expands to **15.14%** (bias **-13.70%**).
  4. **Boundary Elevation at High SoC:** Across 50–90% SoC, Test underprediction is roughly plateaued (-6.8% to -7.2%), but in the 90–100% SoC bin, Test error spikes sharply to **RMSE = 13.99% (MAE = 10.96%, Mean Residual = -9.70%)**.
  5. **Data Range Limitation:** There are **zero test samples below 30% SoC** due to early sub-zero experimental cutoffs.

- **Plausible Physical Interpretations:**
  - *Current & Overpotential Interaction:* The -20°C test data show a strong association between larger initial-SoC residuals and higher discharge current. Increased polarization / internal voltage drop is a plausible physical explanation, but the present dataset does not isolate this mechanism from other confounded effects such as cell aging, SoC dependence, and temperature.
  - *High-SoC Boundary Mechanism:* At near-full charge (90–100% SoC) under extreme cold, steep kinetic overpotentials and phase-transition hysteresis in intercalation materials deviate most severely from the warmer training regime ($\ge 0^\circ\text{C}$).

- **Experimentally Demonstrated Causation (What CANNOT be claimed):**
  - The current experiment does **not** prove that temperature-induced Ohmic drop or capacity fade is the singular cause of the domain shift. Because temperature (-20°C), cumulative cycle aging (40+ prior cycles), and early discharge termination are co-occurring in the physical protocol, they cannot be independently identified from these observational sequences alone.

### 2. Methodological Discipline Reminder
- All quantile trends, correlation values, and regression lines are **purely descriptive diagnostics**.
- No correction model was trained or evaluated.
- The test split remains an unviolated post-hoc benchmark.
